# 04 Predict Ore Masks and Build Intergrowth Descriptors

Use trained binary and multiclass ore segmentation checkpoints to create review artifacts for baseline crops. The binary model is the primary ore outline; the multiclass ore model adds mineral labels clipped to the binary ore mask. Talc prediction is disabled here because talc is a separate UI/user annotation class.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from PIL import Image
import matplotlib.pyplot as plt
import torch

from ore_detection.data.inventory import inventory_baseline_images
from ore_detection.descriptors.intergrowth import summarize_prediction_artifacts, write_descriptor_csv
from ore_detection.inference.model_prediction import load_simple_unet_checkpoint, save_segmentation_prediction

In [ ]:
BASELINE_ROOT = PROJECT_ROOT / 'datasets' / 'baseline'
BINARY_MODEL_SERIAL = '001'
ORE_MODEL_SERIAL = '001'
BINARY_CHECKPOINT = PROJECT_ROOT / 'models' / 'source_binary_segmentation' / BINARY_MODEL_SERIAL / 'best.pt'
ORE_CHECKPOINT = PROJECT_ROOT / 'models' / 'source_ore_segmentation' / ORE_MODEL_SERIAL / 'best.pt'
OUTPUT_ROOT = PROJECT_ROOT / 'data_work' / 'predictions' / 'model_segmentation' / 'baseline_crops'
DESCRIPTOR_CSV = PROJECT_ROOT / 'data_work' / 'descriptors' / 'baseline_intergrowth_descriptors.csv'

RUN_PREDICTION = False
SAMPLE_LIMIT = None  # None means all baseline crop images
BINARY_THRESHOLD = 0.5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'device: {DEVICE}')
print('Talc prediction is disabled in this notebook; talc masks come from reviewed UI annotations.')

## Baseline Crops

`inventory_baseline_images` lists weak-label baseline crops and skips the `panoramas` folder because panoramas are not stored under `Part N/<label>/...`.

In [ ]:
baseline_records = inventory_baseline_images(BASELINE_ROOT)
baseline_records = sorted(baseline_records, key=lambda row: (row['part'], row['label'], row['path']))
print(f'baseline crop count (panoramas skipped): {len(baseline_records)}')
baseline_records[:3]

## Predict Masks

The binary checkpoint creates `ore_mask`, `ore_probability`, and `ore_confidence`. The multiclass checkpoint creates mineral labels and confidence, then the mineral labels are clipped to the binary `ore_mask`.

In [ ]:
if RUN_PREDICTION:
    binary_model = load_simple_unet_checkpoint(BINARY_CHECKPOINT, device=DEVICE)
    ore_model = load_simple_unet_checkpoint(ORE_CHECKPOINT, device=DEVICE)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    records_to_predict = baseline_records if SAMPLE_LIMIT is None else baseline_records[:int(SAMPLE_LIMIT)]
    print(f'prediction input crop count: {len(records_to_predict)} / {len(baseline_records)}')
    for record in records_to_predict:
        image_path = Path(record['path'])
        sample_id = f"{record['part']}_{record['label']}_{image_path.stem}".replace(' ', '_')
        save_segmentation_prediction(
            image_path,
            binary_model=binary_model,
            ore_model=ore_model,
            output_root=OUTPUT_ROOT,
            binary_threshold=BINARY_THRESHOLD,
            sample_id=sample_id,
        )
else:
    print('RUN_PREDICTION is False; using existing prediction artifacts if present.')

## Build Descriptor Table

Descriptors are ore-only and mineral-aware. Talc-derived metrics are intentionally absent until reviewed talc masks exist.

In [ ]:
prediction_dirs = sorted(path for path in OUTPUT_ROOT.glob('*') if (path / 'ore_mask.png').exists()) if OUTPUT_ROOT.exists() else []
descriptor_rows = [summarize_prediction_artifacts(path) for path in prediction_dirs]
write_descriptor_csv(descriptor_rows, DESCRIPTOR_CSV)
print(f'prediction dirs: {len(prediction_dirs)}')
print(f'descriptor rows: {len(descriptor_rows)}')
print(f'descriptor csv: {DESCRIPTOR_CSV}')
descriptor_rows[:3]

## Visual Review

Review raw image, binary ore overlay, binary mask, and clipped multiclass mineral mask before training any weak normal/hard classifier.

In [ ]:
REVIEW_LIMIT = 5
for sample_dir in prediction_dirs[:REVIEW_LIMIT]:
    metadata_path = sample_dir / 'metadata.json'
    metadata = __import__('json').loads(metadata_path.read_text(encoding='utf-8'))
    image_path = Path(metadata['image_path'])
    with Image.open(image_path) as raw, Image.open(sample_dir / 'overlay.png') as overlay, Image.open(sample_dir / 'ore_mask.png') as mask:
        multiclass_path = sample_dir / 'ore_multiclass_mask.png'
        multiclass = Image.open(multiclass_path) if multiclass_path.exists() else None
        fig, axes = plt.subplots(1, 4 if multiclass is not None else 3, figsize=(16, 4))
        axes[0].imshow(raw.convert('RGB'))
        axes[0].set_title('raw')
        axes[1].imshow(overlay.convert('RGB'))
        axes[1].set_title('binary ore overlay')
        axes[2].imshow(mask.convert('L'), cmap='gray')
        axes[2].set_title('binary ore mask')
        if multiclass is not None:
            axes[3].imshow(multiclass.convert('L'), cmap='tab20')
            axes[3].set_title('clipped mineral mask')
            multiclass.close()
        for ax in axes:
            ax.axis('off')
        plt.show()